In [1]:
import os
import cv2
import clip
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

In [7]:
category_name = ["battery", "framework", "hand", "signalinterfaceboard", "stringer", "warehouse"]
data_path = "/workspaces/assemblyhelper/dataset/labels"

In [8]:
category_list = os.listdir(data_path)
print(category_list)

['warehouse', 'stringer', 'hand', 'signalinterfaceboard', 'battery', 'framework']


In [9]:
# Load CLIP
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, preprocess = clip.load("ViT-B/32", device=device)

In [10]:
def pad_to_square(input_image, fill_color=(0, 0, 0)):
    width, height = input_image.size
    new_size = max(width, height)
    new_image = Image.new("RGB", (new_size, new_size), fill_color)
    
    left = (new_size - width) // 2
    top = (new_size - height) // 2    
    
    new_image.paste(input_image, (left, top))
    return new_image

In [11]:
def normalize_feature(feature):
    if len(feature.shape) == 1:
        feature = feature.unsqueeze(0)
        
    min_values, _ = torch.min(feature, dim=1, keepdim=True)
    max_values, _ = torch.max(feature, dim=1, keepdim=True)
    
    feature = (feature - min_values) / (max_values - min_values)
    return feature

In [12]:
images = []
for cate in category_name:
    img_path = os.path.join(data_path, cate)
    img_list = os.listdir(img_path)
    image = []
    for im in img_list:
        image.append(pad_to_square(Image.open(os.path.join(img_path, im))))
    images.append(image)
    
print(images[2])

[<PIL.Image.Image image mode=RGB size=201x201 at 0x7F97E57A3E50>, <PIL.Image.Image image mode=RGB size=287x287 at 0x7F97E567C550>, <PIL.Image.Image image mode=RGB size=334x334 at 0x7F97E567CA90>, <PIL.Image.Image image mode=RGB size=268x268 at 0x7F97E567CB10>, <PIL.Image.Image image mode=RGB size=226x226 at 0x7F97E567CB90>, <PIL.Image.Image image mode=RGB size=223x223 at 0x7F97E567CBD0>, <PIL.Image.Image image mode=RGB size=216x216 at 0x7F97E567CC90>, <PIL.Image.Image image mode=RGB size=161x161 at 0x7F97E567CCD0>]


In [ ]:
images_features = []

cate_features = []
view_vars = []

for image in images:
    preprocessed_img = torch.stack([preprocess(im).to(device) for im in image])
    img_feature = model.encode_image(preprocessed_img)
    img_feature /= img_feature.norm(dim=-1, keepdim=True)
    images_features.append(img_feature)
    
    # 每个类别的均值（view的均值）
    cate_feature = torch.mean(img_feature, dim=0, keepdim=False)
    cate_features.append(cate_feature)
    # 每个类别的方差（view的方差）
    view_var = torch.var(img_feature, dim=0, keepdim=False)
    view_vars.append(view_var)
    
cate_features = torch.stack(cate_features)
view_vars = torch.stack(view_vars)